# 数据建模

## 环境导入

In [1]:
import pickle
import sys
import os
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('default')
sys.path.append(os.path.dirname(os.path.abspath('.')))

from src.models.model_factory import model_train, reorder_columns
from src.utils.all_tolls import create_sample_data
from src.models.model_tools import model_data_clean, sample_select, model_vars_imp, model_features_rfe_select
from src.data.data_clean import *
from src.models.model_report import *
from src.config.feature_config import NOT_MODEL_TRAIN_FEATURE

In [2]:
# 内存情况监控
import psutil
mem_usage = psutil.virtual_memory()
bytes_to_gb = 1024 ** 3

print(f"已使用内存百分比：{mem_usage.percent}%")
print(f"已使用内存：{mem_usage.used / bytes_to_gb:.2f} GB")
print(f"总内存：{mem_usage.total / bytes_to_gb:.2f} GB")

已使用内存百分比：68.6%
已使用内存：10.76 GB
总内存：15.69 GB


## 常量设置

In [3]:
# ---------—---- 参数配置区 --------------------
need_refresh_output_dir = False
y_col_name = "y_flag"
dt_col_name = 'clean_date'
input_data_path = '../input/tc_tc_hj_data.csv'

model_type = "lgb"
# ----------------------------------------------

In [4]:
input_name, _ = os.path.splitext(os.path.basename(input_data_path))
output_path = Path('../output')
    
if need_refresh_output_dir and output_path.exists():
    shutil.rmtree(output_path)
    print(f"已清空目录: {output_path}")
need_refresh_output_dir
output_dir = f'../output/{input_name}'
os.makedirs(output_dir, exist_ok=True)

## 1-数据清洗

### 1-1 数据读取

In [7]:
Y_data = pd.read_csv(input_data_path)
Y_data = data_time_clean(Y_data,dt_col_name)

2026-06-08 17:41:43 | INFO     | 时间变量处理开始
2026-06-08 17:41:47 | INFO     | 	> 待处理时间的列为: clean_date, 处理后时间列为: clean_date, 月份列为: clean_yearmonth
2026-06-08 17:41:47 | INFO     | 时间变量处理结束


In [8]:
cols_to_drop = [col for col in Y_data.columns if col.startswith('P') and '_' in col]
Y_data = Y_data.drop(columns=cols_to_drop)

In [16]:
d_df = pd.read_excel('../output/tc_tc_hj_data/数据字典to智策.xlsx', sheet_name='M1013')
exclude_names = d_df[d_df['含义'].str.contains('距今天数|迄今天数', na=False)]['名称'].tolist()
filtered_ls = [field for field in Y_data.columns if field not in exclude_names]

In [17]:
Y_data = Y_data[filtered_ls]

In [18]:
# 建模数据清洗并导出
NOT_MODEL_TRAIN_FEATURE.extend([y_col_name, dt_col_name, 'clean_date', 'clean_yearmonth'])
# 缺失率处理
model_data1 = calculate_missing_rate(Y_data ,NOT_MODEL_TRAIN_FEATURE )
# 同值率处理
model_data2 = drop_same_rate_high_column(model_data1 ,NOT_MODEL_TRAIN_FEATURE)
# 空值统一替换
model_data = nan_data_replace(model_data2,[y_col_name])
model_data.to_pickle(f"{output_dir}/[data]_clean_input_data.pkl")

2026-06-08 18:01:09 | INFO     | > 缺失率处理开始


2026-06-08 18:01:10 | INFO     | 	> 原始数据集: 行数 297693, 列数 1006
2026-06-08 18:01:10 | INFO     | 	> 删除缺失率大于80.0%的值，共计删除 10 列
2026-06-08 18:01:10 | INFO     | 	> 删除列为: ['M1013_0892', 'M1013_6574', 'M1013_1097', 'M1013_5144', 'M1013_2927', 'M1013_8702', 'M1013_7939', 'M1013_7364', 'M1013_9806', 'M1013_0400']
2026-06-08 18:01:10 | INFO     | 	> 清洗数据集: 行数 297693, 列数 996
2026-06-08 18:01:10 | INFO     | 缺失率处理结束
2026-06-08 18:01:10 | INFO     | 同值率处理开始
	> 处理中 |██████████████████████████████████████████████████| 100.0% 
2026-06-08 18:01:22 | INFO     | 	> 原始数据集: 行数 297693, 列数 996
2026-06-08 18:01:22 | INFO     | 	> 删除同值率大于80.0%的值, 共计删除 28 列
2026-06-08 18:01:22 | INFO     | 	> 删除列为: ['M1013_6032', 'M1013_5471', 'M1013_3671', 'M1013_1398', 'M1013_2922', 'M1013_5108', 'M1013_6903', 'M1013_1531', 'M1013_0692', 'M1013_1955', 'M1013_1873', 'M1013_8273', 'M1013_4398', 'M1013_0645', 'M1013_0877', 'M1013_7130', 'M1013_2350', 'M1013_1757', 'M1013_5149', 'M1013_4700', 'M1013_6207', 'M1013_4907', 'M1013_07

## 2-模型训练

### 2-1 模型训练准备

In [19]:
# output_dir = f'../output/{input_name}/'
# model_data = pd.read_pickle(f"{output_dir}/clean_input_data.pkl")

model_data, model_vars = model_data_clean(model_data,y_col_name,NOT_MODEL_TRAIN_FEATURE)
monthly_y_distribution(model_data, y_col_name)

2026-06-08 18:01:46 | INFO     | 总变量数: 968
2026-06-08 18:01:46 | INFO     | 建模变量数: 964
2026-06-08 18:01:46 | INFO     | 其他变量数: 4
2026-06-08 18:01:46 | INFO     | 其他变量: ['map_key', 'clean_date', 'clean_yearmonth', 'y_flag']


,年月,总样本数,坏样本数,好样本数,样本占比,坏样本率,好样本率,PSI
0,202501,34794,1889,32905,11.69%,5.43%,94.57%,0.000000
1,202502,40545,1919,38626,13.62%,4.73%,95.27%,0.001006
2,202503,39235,1884,37351,13.18%,4.80%,95.20%,0.000812
3,202504,63833,1316,62517,21.44%,2.06%,97.94%,0.033785
4,202505,62122,1242,60880,20.87%,2.00%,98.00%,0.035485
5,202506,29558,846,28712,9.93%,2.86%,97.14%,0.017121
6,202507,27606,575,27031,9.27%,2.08%,97.92%,0.033221


In [20]:
oot_list = ['202506','202507']
model_data, data_info = sample_select(model_data,y_col_name,oot_list)
model_data.to_pickle(f"{output_dir}/[data]_train_input_data.pkl")

2026-06-08 18:02:00 | INFO     | 开始数据分割
2026-06-08 18:02:00 | INFO     | 	> 分割参数: test_size: 0.2, random_state: 100
2026-06-08 18:02:01 | INFO     | 	> oot数据集为: ['202506', '202507']
2026-06-08 18:02:10 | INFO     | 数据分割完成


### 2-2 初次训练

In [21]:
model_info = model_train(model_data,model_vars,y_col_name,model_type,'bys')
model_info_df = reorder_columns(pd.DataFrame(model_info))
model_info_df

2026-06-08 18:02:26 | INFO     | 开始模型训练
2026-06-08 18:02:26 | INFO     | 	> 模型名称: lgb
2026-06-08 18:02:26 | INFO     | 	> 模型调参方式: bys
2026-06-08 18:02:26 | INFO     | 	> 模型参数范围为: {'n_estimators': [100, 200, 300, 400, 500], 'learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2], 'max_depth': [3, 4, 5, 6, 7], 'min_child_samples': [20, 50, 100, 150, 200], 'subsample': [0.6, 0.7, 0.8, 0.9, 1.0], 'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0], 'reg_alpha': [0, 1, 5, 10, 20], 'reg_lambda': [0, 1, 5, 10], 'max_bin': [100, 150, 200, 250, 300]}
2026-06-08 18:02:26 | INFO     | 	> 入模变量数量为: 964
2026-06-08 18:02:28 | INFO     | 贝叶斯优化：共 10 轮迭代，搜索空间: ['n_estimators_idx', 'learning_rate_idx', 'max_depth_idx', 'min_child_samples_idx', 'subsample_idx', 'colsample_bytree_idx', 'reg_alpha_idx', 'reg_lambda_idx', 'max_bin_idx']

2026-06-08 18:02:28 | INFO     | 第 1 次贝叶斯目标优化
2026-06-08 18:03:17 | INFO     | ============================================================
2026-06-08 18:03:17 | INFO     | 参数配置:
2026-06-0

,params,final_score,is_overfit,train_ks,test_ks,vldt_ks,train_auc,test_auc,vldt_auc,train_test_psi,train_vldt_psi,train_test_vldt_psi,model
0,"{'n_estimators': 200, 'learning_rate': 0.005, ...",0.316736,False,0.331,0.300,0.287,0.724,0.698,0.682,0.0001,0.0003,0.0002,"LGBMClassifier(colsample_bytree=0.9, learning_..."
1,"{'n_estimators': 200, 'learning_rate': 0.005, ...",0.316736,False,0.331,0.300,0.287,0.724,0.698,0.682,0.0001,0.0003,0.0002,"LGBMClassifier(colsample_bytree=0.9, learning_..."
2,"{'n_estimators': 100, 'learning_rate': 0.005, ...",0.309720,False,0.308,0.294,0.280,0.710,0.693,0.678,0.0000,0.0000,0.0000,"LGBMClassifier(learning_rate=0.005, max_bin=10..."
3,"{'n_estimators': 100, 'learning_rate': 0.005, ...",0.308700,False,0.284,0.294,0.279,0.686,0.688,0.675,0.0000,0.0000,0.0000,"LGBMClassifier(learning_rate=0.005, max_bin=10..."
4,"{'n_estimators': 100, 'learning_rate': 0.005, ...",0.307720,False,0.305,0.292,0.278,0.711,0.692,0.678,0.0000,0.0000,0.0000,"LGBMClassifier(learning_rate=0.005, max_bin=30..."
5,"{'n_estimators': 100, 'learning_rate': 0.005, ...",0.305620,False,0.309,0.295,0.276,0.713,0.691,0.676,0.0000,0.0000,0.0000,"LGBMClassifier(learning_rate=0.005, max_bin=30..."
6,"{'n_estimators': 100, 'learning_rate': 0.005, ...",0.305580,False,0.311,0.297,0.276,0.709,0.694,0.676,0.0000,0.0000,0.0000,"LGBMClassifier(colsample_bytree=0.8, learning_..."
7,"{'n_estimators': 500, 'learning_rate': 0.005, ...",0.171020,True,0.380,0.307,0.283,0.758,0.702,0.681,0.0000,0.0021,0.0020,"LGBMClassifier(colsample_bytree=0.7, learning_..."
8,"{'n_estimators': 200, 'learning_rate': 0.05, '...",0.160464,True,0.443,0.313,0.263,0.804,0.704,0.676,0.0009,0.0031,0.0026,"LGBMClassifier(learning_rate=0.05, max_bin=300..."
9,"{'n_estimators': 200, 'learning_rate': 0.2, 'm...",0.131680,True,0.670,0.271,0.207,0.919,0.681,0.636,0.0135,0.0164,0.0122,"LGBMClassifier(colsample_bytree=0.7, learning_..."


### 2-3 特征筛选

In [31]:
model_choose = 0   # !!!选择需要的模型编号
result_model_vas = model_vars_imp(model_info[model_choose].get("model"), method="auto", rate=0.65)
result_model_parms = model_info[model_choose].get("params")

2026-06-09 09:27:46 | INFO     | 变量的选择为：auto
2026-06-09 09:27:46 | INFO     | 模型变量筛选重要性占比65.0%的数为： 141


In [32]:
model_info_re = model_train(model_data,result_model_vas,y_col_name,model_type,'bys')
model_info_df_re = reorder_columns(pd.DataFrame(model_info_re))
model_info_df_re

2026-06-09 09:27:53 | INFO     | 开始模型训练
2026-06-09 09:27:53 | INFO     | 	> 模型名称: lgb
2026-06-09 09:27:53 | INFO     | 	> 模型调参方式: bys
2026-06-09 09:27:53 | INFO     | 	> 模型参数范围为: {'n_estimators': [100, 200, 300, 400, 500], 'learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2], 'max_depth': [3, 4, 5, 6, 7], 'min_child_samples': [20, 50, 100, 150, 200], 'subsample': [0.6, 0.7, 0.8, 0.9, 1.0], 'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0], 'reg_alpha': [0, 1, 5, 10, 20], 'reg_lambda': [0, 1, 5, 10], 'max_bin': [100, 150, 200, 250, 300]}
2026-06-09 09:27:53 | INFO     | 	> 入模变量数量为: 141
2026-06-09 09:27:54 | INFO     | 贝叶斯优化：共 10 轮迭代，搜索空间: ['n_estimators_idx', 'learning_rate_idx', 'max_depth_idx', 'min_child_samples_idx', 'subsample_idx', 'colsample_bytree_idx', 'reg_alpha_idx', 'reg_lambda_idx', 'max_bin_idx']

2026-06-09 09:27:54 | INFO     | 第 1 次贝叶斯目标优化
2026-06-09 09:28:02 | INFO     | ============================================================
2026-06-09 09:28:02 | INFO     | 参数配置:
2026-06-0

,params,final_score,is_overfit,train_ks,test_ks,vldt_ks,train_auc,test_auc,vldt_auc,train_test_psi,train_vldt_psi,train_test_vldt_psi,model
0,"{'n_estimators': 200, 'learning_rate': 0.005, ...",0.313672,False,0.327,0.300,0.284,0.723,0.696,0.680,0.0002,0.0011,0.0010,"LGBMClassifier(colsample_bytree=0.9, learning_..."
1,"{'n_estimators': 200, 'learning_rate': 0.005, ...",0.311656,False,0.329,0.299,0.282,0.721,0.697,0.680,0.0001,0.0005,0.0005,"LGBMClassifier(colsample_bytree=0.9, learning_..."
2,"{'n_estimators': 100, 'learning_rate': 0.005, ...",0.309680,False,0.307,0.296,0.280,0.709,0.693,0.678,0.0000,0.0000,0.0000,"LGBMClassifier(learning_rate=0.005, max_bin=15..."
3,"{'n_estimators': 100, 'learning_rate': 0.005, ...",0.309620,False,0.306,0.299,0.280,0.708,0.696,0.678,0.0000,0.0000,0.0000,"LGBMClassifier(colsample_bytree=0.7, learning_..."
4,"{'n_estimators': 100, 'learning_rate': 0.005, ...",0.309580,False,0.307,0.301,0.280,0.705,0.695,0.677,0.0000,0.0000,0.0000,"LGBMClassifier(colsample_bytree=0.6, learning_..."
5,"{'n_estimators': 100, 'learning_rate': 0.005, ...",0.308640,False,0.305,0.297,0.279,0.708,0.695,0.678,0.0000,0.0000,0.0000,"LGBMClassifier(colsample_bytree=0.8, learning_..."
6,"{'n_estimators': 500, 'learning_rate': 0.005, ...",0.169460,True,0.362,0.307,0.280,0.747,0.701,0.681,0.0000,0.0025,0.0025,"LGBMClassifier(colsample_bytree=0.7, learning_..."
7,"{'n_estimators': 300, 'learning_rate': 0.1, 'm...",0.159756,True,0.382,0.298,0.261,0.762,0.700,0.670,0.0001,0.0011,0.0010,"LGBMClassifier(colsample_bytree=0.9, max_bin=2..."
8,"{'n_estimators': 200, 'learning_rate': 0.05, '...",0.158032,True,0.417,0.306,0.258,0.784,0.702,0.670,0.0002,0.0016,0.0015,"LGBMClassifier(learning_rate=0.05, max_bin=300..."
9,"{'n_estimators': 200, 'learning_rate': 0.2, 'm...",0.140572,True,0.617,0.283,0.224,0.893,0.687,0.644,0.0062,0.0105,0.0082,"LGBMClassifier(colsample_bytree=0.7, learning_..."


In [33]:
result_model_parms = model_info_re[0].get("params")

### 2-4 最终模型

In [34]:
model_info_final = model_train(model_data,result_model_vas,y_col_name,model_type,model_params=result_model_parms)
result_model = model_info_final[0].get("model")

with open(f'{output_dir}/[model]_{input_name}_{model_type}.pkl', 'wb') as f:
    pickle.dump(result_model, f)

2026-06-09 09:30:25 | INFO     | 开始模型训练
2026-06-09 09:30:25 | INFO     | 	> 模型名称: lgb
2026-06-09 09:30:25 | INFO     | 	> 模型调参方式: default
2026-06-09 09:30:25 | INFO     | 	> 模型参数范围为: {'n_estimators': 200, 'learning_rate': 0.005, 'max_depth': 7, 'min_child_samples': 50, 'subsample': 0.8, 'colsample_bytree': 0.9, 'reg_alpha': 0, 'reg_lambda': 0, 'max_bin': 300}
2026-06-09 09:30:25 | INFO     | 	> 入模变量数量为: 141
2026-06-09 09:30:42 | INFO     | ============================================================
2026-06-09 09:30:42 | INFO     | 参数配置:
2026-06-09 09:30:42 | INFO     |    n_estimators: 200
2026-06-09 09:30:42 | INFO     |    learning_rate: 0.005
2026-06-09 09:30:42 | INFO     |    max_depth: 7
2026-06-09 09:30:42 | INFO     |    min_child_samples: 50
2026-06-09 09:30:42 | INFO     |    subsample: 0.8
2026-06-09 09:30:42 | INFO     |    colsample_bytree: 0.9
2026-06-09 09:30:42 | INFO     |    reg_alpha: 0
2026-06-09 09:30:42 | INFO     |    reg_lambda: 0
2026-06-09 09:30:42 | INFO    

## 3-模型报告

In [36]:
clean_in_data = pd.read_pickle(f"{output_dir}/[data]_clean_input_data.pkl")
clean_in_data['probability(1)'] = result_model.predict_proba(clean_in_data[result_model.feature_name_])[:, 1]
clean_in_data.to_csv(f"{output_dir}/[data]_clean_input_data_p1.csv", index=False)

In [35]:
report_info = {
    "模型类型": model_type,
    "数据集名称": input_name,
    "y 标列名": y_col_name,
    "vldt 年月集": ', '.join(oot_list)
}

report = model_report_main(
    model=result_model,
    model_info=report_info,
    model_data=model_data,
    y_flag=y_col_name,
    ym_flag='clean_yearmonth',
    data_flag='samp_type',
    output_dir=output_dir,
    output_excel=f'{output_dir}/模型评估报告.xlsx',
    pkl_path=f'{output_dir}/[model]_{input_name}_{model_type}.pkl'
)

✅ 样本模型综合效果报告已生成
✅ 样本入模变量信息报告已生成
✅ 样本变量分箱_是否单调报告已生成
✅ 模型稳定性报告已生成
✅ 变量相关性分析报告已生成
✅ 变量分箱图表数据已生成
✅ 模型报告已导出至: ../output/tc_tc_hj_data/模型评估报告.xlsx
✅ 临时文件已保存至: ../output/tc_tc_hj_data\tmp
✅ PMML文件已导出: ../output/tc_tc_hj_data/[model]_tc_tc_hj_data_lgb.pmml
✅ OLE对象已插入: ../output/tc_tc_hj_data/[model]_tc_tc_hj_data_lgb.pkl -> B13
✅ OLE对象已插入: ../output/tc_tc_hj_data/[model]_tc_tc_hj_data_lgb.pmml -> B13
